# Text-to-SQL with Vanna.ai (SQLite Version)

**Duration:** 1-2 hours  
**Level:** Intermediate  
**Framework:** Vanna.ai 2.0 Agent Framework  
**Database:** SQLite (Local)

---

## Overview

This tutorial demonstrates how to build a Text-to-SQL agent using Vanna.ai and a local SQLite database. This version removes the need for an external PostgreSQL server.

## Prerequisites

- OpenAI API key
- Python 3.12+

In [ ]:
# Install required packages
!pip install vanna openai pandas sqlalchemy -q

## 1. Import Libraries

In [ ]:
# Core Vanna 2.0 imports
from vanna import Agent
from vanna.integrations.openai import OpenAILlmService
from vanna.integrations.sqlite import SQLiteRunner
from vanna.core.registry import ToolRegistry
from vanna.tools import RunSqlTool
from vanna.core.user import UserResolver, User, RequestContext
from vanna.integrations.local.agent_memory import DemoAgentMemory

# Standard libraries
import os
import sqlite3
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

print("✓ Libraries imported successfully")

## 2. Setup SQLite Database

We will create a local SQLite database and populate it with sample e-commerce data.

In [ ]:
# Create/Connect to local SQLite database
db_filename = 'ecommerce.sqlite'
conn = sqlite3.connect(db_filename)
cursor = conn.cursor()

# 1. Create Tables
cursor.executescript("""
    DROP TABLE IF EXISTS orders;
    DROP TABLE IF EXISTS customers;
    DROP TABLE IF EXISTS products;

    CREATE TABLE customers (
        id INTEGER PRIMARY KEY,
        name TEXT,
        email TEXT,
        segment TEXT,
        country TEXT
    );

    CREATE TABLE products (
        id INTEGER PRIMARY KEY,
        name TEXT,
        category TEXT,
        price REAL,
        stock_quantity INTEGER
    );

    CREATE TABLE orders (
        id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        order_date DATE,
        total_amount REAL,
        status TEXT,
        FOREIGN KEY (customer_id) REFERENCES customers (id)
    );
""")

# 2. Insert Dummy Data
customers_data = [
    (1, 'Alice Smith', 'alice@example.com', 'SMB', 'USA'),
    (2, 'Bob Jones', 'bob@example.com', 'Enterprise', 'Canada'),
    (3, 'Charlie Brown', 'charlie@example.com', 'Individual', 'UK'),
    (4, 'Diana Prince', 'diana@example.com', 'Enterprise', 'USA'),
    (5, 'Evan Wright', 'evan@example.com', 'SMB', 'USA')
]

products_data = [
    (1, 'Laptop Pro', 'Electronics', 1200.00, 50),
    (2, 'Ergo Chair', 'Furniture', 300.00, 100),
    (3, 'Wireless Mouse', 'Electronics', 25.00, 200),
    (4, 'CRM License', 'Software', 500.00, 1000),
    (5, 'Monitor 4K', 'Electronics', 400.00, 30)
]

orders_data = [
    (1, 1, '2023-01-15', 1225.00, 'Delivered'),   # Alice bought Laptop + Mouse
    (2, 2, '2023-02-01', 5000.00, 'Processing'),  # Bob bought 10 CRM Licenses
    (3, 4, '2023-02-10', 1600.00, 'Delivered'),   # Diana bought Laptop + Monitor
    (4, 1, '2023-03-05', 300.00, 'Shipped'),      # Alice bought Chair
    (5, 3, '2023-03-10', 25.00, 'Delivered')      # Charlie bought Mouse
]

cursor.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", customers_data)
cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products_data)
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders_data)

conn.commit()
print("✓ SQLite database created and populated")

## 3. Initialize Vanna Agent

In [ ]:
# Load API Key
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    # Fallback for tutorial (User should replace this)
    print("⚠️ OPENAI_API_KEY not found in env. Please set it manually below if needed.")
    # OPENAI_API_KEY = "sk-..."

# 1. Initialize LLM
llm = OpenAILlmService(
    api_key=OPENAI_API_KEY,
    model="gpt-4o"
)

# 2. Initialize SQLite Runner
# Note: We pass the dict config expected by SQLiteRunner, or just the db path depending on implementation
# Checking Vanna docs/code, SQLiteRunner usually takes a config dict: {'database': 'path/to/db'}
sqlite_runner = SQLiteRunner(
    config={'database': db_filename}
)

# 3. Register Tools
tools = ToolRegistry()
tools.register_local_tool(
    RunSqlTool(sql_runner=sqlite_runner),
    access_groups=['user', 'admin']
)

# 4. User Resolver
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        return User(
            id="tutorial_user",
            email="student@tutorial.com",
            group_memberships=['user', 'admin']
        )

user_resolver = SimpleUserResolver()

# 5. Create Agent
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=user_resolver,
    agent_memory=DemoAgentMemory()
)

print("✓ Vanna Agent initialized with SQLite")

## 4. Provide Schema Context

In [ ]:
SCHEMA_CONTEXT = """
DATABASE SCHEMA (SQLite):

Table: customers
Columns: id, name, email, segment, country

Table: products
Columns: id, name, category, price, stock_quantity

Table: orders
Columns: id, customer_id, order_date, total_amount, status

Relationships:
- customers.id -> orders.customer_id
"""
print("✓ Schema context defined")

## 5. Ask Questions

In [ ]:
async def ask_agent(question: str):
    full_message = f"{SCHEMA_CONTEXT}\n\nQUESTION: {question}"
    request_context = RequestContext()
    
    print(f"🤔 {question}\n")
    
    async for component in agent.send_message(
        request_context=request_context,
        message=full_message
    ):
        rich_comp = component.rich_component
        if hasattr(rich_comp, 'rows') and rich_comp.rows:
            df = pd.DataFrame(rich_comp.rows)
            display(df)
        elif hasattr(rich_comp, 'text') and rich_comp.text:
            print(f"💬 {rich_comp.text}\n")
        elif hasattr(rich_comp, 'sql') and rich_comp.sql:
             print(f"🔍 SQL: {rich_comp.sql}\n")

# Test 1
await ask_agent("How many customers are in the database?")

In [ ]:
# Test 2
await ask_agent("Show me the available products")

In [ ]:
# Test 3
await ask_agent("What is the total revenue from Delivered orders?")